In [1]:
!pip install xlrd


In [34]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score
)

In [36]:
# Load Dataset
df = pd.read_excel("default of credit card clients.xls", header=1)

# Remove ID
df.drop("ID", axis=1, inplace=True)

# Target Column
X = df.drop("default payment next month", axis=1)
y = df["default payment next month"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
# Scale Data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [44]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "KNN":
        KNeighborsClassifier(n_neighbors=5),

    "Naive Bayes":
        GaussianNB(),

    "Random Forest":
      RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
}


In [46]:
results = []

for name, model in models.items():

    if name in ["Logistic Regression", "KNN"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        prob = model.predict_proba(X_test_scaled)[:,1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        if hasattr(model,'predict_proba'):
            prob = model.predict_proba(X_test)[:,1]
        else:
            prob = preds

    accuracy = accuracy_score(y_test,preds)
    precision = precision_score(y_test,preds)
    recall = recall_score(y_test,preds)
    f1 = f1_score(y_test,preds)
    auc = roc_auc_score(y_test,prob)
    mcc = matthews_corrcoef(y_test,preds)

    results.append([
        name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

    file_name = name.lower().replace(" ","_") + ".pkl"
    joblib.dump(model, f"{file_name}")

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

results_df.to_csv("metrics.csv", index=False)

X_test.assign(
    Actual=y_test.values
).to_csv(
    "test_data.csv",
    index=False
)

print(results_df)

                 Model  Accuracy       AUC  Precision    Recall        F1  \
0  Logistic Regression  0.807667  0.707636   0.686825  0.239638  0.355307   
1        Decision Tree  0.714500  0.607451   0.369418  0.411454  0.389305   
2                  KNN  0.792833  0.701435   0.548724  0.356443  0.432161   
3          Naive Bayes  0.416000  0.651567   0.249597  0.817634  0.382446   
4        Random Forest  0.817667  0.771570   0.666191  0.351922  0.460552   

        MCC  
0  0.324443  
1  0.204216  
2  0.323267  
3  0.111087  
4  0.389999  
